# Task 01: Generate NER Dataset with LLM

Use `gpt-4o-mini` to annotate cybersecurity texts for GLiNER training.

Entity types: `malware`, `vulnerability`, `attack_technique`, `affected_software`, `threat_actor`

You will:
1. Call LLM to extract entities from a text, parse the `entity_text <> entity_type` response
2. Implement `build_gliner_example(client, text)` — full annotation pipeline
3. Generate a dataset from all 20 reports and validate it

In [ ]:
import re, json, os
from openai import OpenAI
from collections import Counter

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual key
client = OpenAI(api_key=api_key)

ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]

def tokenize(text):
    """GLiNER-compatible tokenizer."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

print(f"✓ Loaded {len(reports)} reports")

## Part 1: Call LLM and Parse Response

Write a system prompt that instructs the model to output one entity per line as:
`entity_text <> entity_type`

Implement:
- `call_llm_ner(client, text)` → raw string response
- `parse_ner_output(raw_output)` → `list[(entity_text, entity_type)]`

Store the results for the first report in `raw_output` and `entities`.

In [ ]:
# YOUR CODE HERE
test_text = reports[0]['text']

# SYSTEM_PROMPT = ...

# def call_llm_ner(client, text):
#     ...

# def parse_ner_output(raw_output):
#     ...

# raw_output = ...
# entities = ...


In [ ]:
# TEST — Part 1
assert 'raw_output' in vars(), "Store LLM response in 'raw_output'"
assert isinstance(raw_output, str) and len(raw_output) > 0
assert 'entities' in vars(), "Store parsed entities in 'entities'"
assert isinstance(entities, list), "entities must be a list"
assert len(entities) >= 2, f"Expected at least 2 entities for report 0, got {len(entities)}"
for ent_text, ent_type in entities:
    assert isinstance(ent_text, str) and len(ent_text) > 0
    assert ent_type in ENTITY_TYPES, f"Unknown entity type: {ent_type!r}"
# Spot check: LockBit or CVE should be detected
entity_texts_lower = [e.lower() for e, _ in entities]
assert any('lockbit' in e or 'cve' in e or 'citrix' in e for e in entity_texts_lower), \
    f"Expected LockBit/CVE/Citrix in entities, got: {entities}"
print(f"✓ Parsed {len(entities)} entities:")
for e_text, e_type in entities:
    print(f"  [{e_type}] '{e_text}'")

## Part 2: Implement `build_gliner_example`

Implement `find_token_span(tokenized_text, entity_text)` that finds the token-level span
(start, end) inclusive, then implement `build_gliner_example(client, text)` that returns:
```python
{"tokenized_text": [...], "ner": [[start, end, label], ...]}
```

In [ ]:
# YOUR CODE HERE
def find_token_span(tokenized_text, entity_text):
    """
    Find token-level span of entity_text in tokenized_text.
    Returns (start_idx, end_idx) inclusive, or None if not found.
    """
    pass


def build_gliner_example(client, text):
    """
    Annotate a single text using LLM.
    Returns {tokenized_text, ner} or None if no entities found.
    """
    pass


In [ ]:
# TEST — Part 2
example = build_gliner_example(client, reports[0]['text'])
assert example is not None, "build_gliner_example returned None for report 0"
assert 'tokenized_text' in example and 'ner' in example
assert isinstance(example['tokenized_text'], list) and len(example['tokenized_text']) > 5
assert isinstance(example['ner'], list) and len(example['ner']) >= 2

n_tokens = len(example['tokenized_text'])
for span in example['ner']:
    assert len(span) == 3, f"Span must be [start, end, label], got {span}"
    s, e, label = span
    assert 0 <= s <= e < n_tokens, f"Invalid span [{s},{e}] for {n_tokens} tokens"
    assert label in ENTITY_TYPES, f"Unknown label: {label}"

print(f"✓ build_gliner_example works")
print(f"  Tokens: {len(example['tokenized_text'])} | Entities: {len(example['ner'])}")
for s, e, label in example['ner']:
    entity_surface = ' '.join(example['tokenized_text'][s:e+1])
    print(f"  [{label}] span=[{s},{e}] → '{entity_surface}'")

## Part 3: Generate Full Dataset

Implement `build_ner_dataset(client, reports)` and annotate all 20 reports.
Store the result in `ner_dataset`. Validate and save to `fixtures/expected/ner_dataset.json`.

Requirements:
- At least 15 valid examples (some reports may yield no entities)
- All spans within bounds
- All entity types from `ENTITY_TYPES`
- At least 3 different entity types represented in the dataset

In [ ]:
# YOUR CODE HERE
from tqdm import tqdm

# def build_ner_dataset(client, reports):
#     ...

# ner_dataset = ...


In [ ]:
# TEST — Part 3
assert 'ner_dataset' in vars(), "Store dataset in 'ner_dataset'"
assert len(ner_dataset) >= 15, f"Expected ≥15 examples, got {len(ner_dataset)}"

all_labels_used = set()
for ex in ner_dataset:
    assert 'tokenized_text' in ex and 'ner' in ex
    n = len(ex['tokenized_text'])
    for s, e, label in ex['ner']:
        assert 0 <= s <= e < n, f"Invalid span [{s},{e}] in {n}-token text"
        assert label in ENTITY_TYPES
        all_labels_used.add(label)

assert len(all_labels_used) >= 3, f"Need ≥3 entity types, got: {all_labels_used}"

# Save
save_path = os.path.normpath(os.path.join(fixtures, "..", "expected", "ner_dataset.json"))
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(ner_dataset, f, indent=2)

label_counts = Counter(span[2] for ex in ner_dataset for span in ex['ner'])
print(f"✓ Generated {len(ner_dataset)} NER training examples")
print(f"  Entity types found: {sorted(all_labels_used)}")
print(f"  Entity distribution: {dict(label_counts.most_common())}")
print(f"  Saved to fixtures/expected/ner_dataset.json")